# Simulación TVB de una región, paso a paso

Este cuaderno reproduce el ejemplo simplificado sin leer JSON ni datos externos. Usamos el modelo `ZerlautAdaptationFirstOrder` de TVB, un pulso determinista y una sola región sin conexiones.

El flujo es: **constantes → estímulo → modelo → estado inicial → integración de Heun → conversión de unidades → gráfico y NPZ**.

## 1. Importaciones y entradas hardcodeadas

TVB representa las tasas de disparo del modelo Zerlaut en kHz. Para que las entradas y salidas sean más fáciles de leer, aquí se expresan en Hz y sólo se convierten en la frontera con TVB.

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tvb.simulator.models.zerlaut import ZerlautAdaptationFirstOrder

RUN = 0
DURATION_MS = 2000.0
DT_MS = 0.5
BASELINE_HZ = 5.0
PULSE_AMPLITUDE_HZ = 40.0
PULSE_ONSET_MS = 500.0
PULSE_DURATION_MS = 700.0
INPUT_SCALE = 1.0
FI_RATIO = 0.5

## 2. Cuadrícula temporal y pulso

El estímulo vale 5 Hz como línea base y suma 40 Hz entre 500 y 1200 ms. Siguiendo la convención del proyecto de referencia:

- `Fe_ext(t) = INPUT_SCALE × stimulus(t)`
- `Fi_ext(t) = FI_RATIO × Fe_ext(t)`

In [ ]:
time_ms = np.arange(0.0, DURATION_MS, DT_MS)
stimulus_hz = np.full(time_ms.shape, BASELINE_HZ)
pulse = (time_ms >= PULSE_ONSET_MS) & (time_ms < PULSE_ONSET_MS + PULSE_DURATION_MS)
stimulus_hz[pulse] += PULSE_AMPLITUDE_HZ

fe_ext_hz = INPUT_SCALE * stimulus_hz
fi_ext_hz = FI_RATIO * fe_ext_hz

plt.figure(figsize=(10, 3))
plt.plot(time_ms, stimulus_hz, color='black', label='estímulo')
plt.plot(time_ms, fi_ext_hz, label='Fi externa')
plt.xlabel('Tiempo (ms)')
plt.ylabel('Hz')
plt.grid(alpha=0.25)
plt.legend();

## 3. Modelo Zerlaut-AdEx de TVB

Configuramos las propiedades celulares, sinápticas y de población. Cada valor se almacena como un array porque TVB permite parametrizar distintas regiones, aunque aquí sólo exista una.

In [ ]:
model_parameters = {
    'g_L': 10.0, 'C_m': 200.0, 'E_L_e': -60.0, 'E_L_i': -65.0,
    'a_e': 4.0, 'b_e': 60.0, 'tau_w_e': 200.0,
    'a_i': 0.0, 'b_i': 20.0, 'tau_w_i': 100.0,
    'E_e': 0.0, 'E_i': -80.0, 'Q_e': 1.0, 'Q_i': 5.0,
    'tau_e': 5.0, 'tau_i': 5.0, 'N_tot': 10000,
    'p_connect_e': 0.05, 'p_connect_i': 0.05, 'g': 0.2,
    'K_ext_e': 400, 'K_ext_i': 35, 'T': 10.0,
}
integer_parameters = {'N_tot', 'K_ext_e', 'K_ext_i'}

model = ZerlautAdaptationFirstOrder()
for name, value in model_parameters.items():
    dtype = np.int64 if name in integer_parameters else np.float64
    setattr(model, name, np.array([value], dtype=dtype))
model.weight_noise = np.array([0.0])  # simulación reproducible
model.configure()

model.state_variables

## 4. Estado y acoplamiento de una sola región

El estado tiene cinco variables (`E`, `I`, `W_e`, `W_i`, deriva OU), una región y un modo: forma `(5, 1, 1)`.

**No se carga un fichero de conectividad.** El tensor `coupling` es cero, por lo que no entra señal de otra región ni hay autoacoplamiento.

In [ ]:
state = np.zeros((5, 1, 1), dtype=np.float64)
coupling = np.zeros((1, 1, 1), dtype=np.float64)
history = np.zeros((time_ms.size, 5), dtype=np.float64)

print('estado:', state.shape, '| acoplamiento:', coupling.shape, '| pasos:', time_ms.size)

## 5. Integración temporal con Heun

Para cada paso: (1) cargamos las entradas E/I en el modelo, (2) calculamos una primera pendiente, (3) predecimos el estado al final del paso, (4) calculamos la segunda pendiente y (5) promediamos ambas.

Los cuatro nombres `external_input_…` indican el tipo de entrada (E o I) y la población diana (E o I).

In [ ]:
def set_external_drive(model, fe_hz, fi_hz):
    fe_khz = np.array([fe_hz / 1000.0])
    fi_khz = np.array([fi_hz / 1000.0])
    model.external_input_ex_ex = fe_khz
    model.external_input_in_ex = fe_khz.copy()
    model.external_input_ex_in = fi_khz
    model.external_input_in_in = fi_khz.copy()

for index in range(time_ms.size - 1):
    set_external_drive(model, fe_ext_hz[index], fi_ext_hz[index])
    slope_1 = model.dfun(state, coupling)

    predicted_state = state + DT_MS * slope_1
    set_external_drive(model, fe_ext_hz[index + 1], fi_ext_hz[index + 1])
    slope_2 = model.dfun(predicted_state, coupling)

    state = state + 0.5 * DT_MS * (slope_1 + slope_2)
    state[0:2] = np.clip(state[0:2], 0.0, 5.0)
    state[2:4] = np.clip(state[2:4], -100000.0, 100000.0)
    state[4] = 0.0
    history[index + 1] = state[:, 0, 0]



## 6. Conversión a Hz y visualización

Las tasas E/I salen de TVB en kHz; se multiplican por 1000. Las adaptaciones ya están en pA.

In [ ]:
E_hz = history[:, 0] * 1000.0
I_hz = history[:, 1] * 1000.0
W_e_pA = history[:, 2]
W_i_pA = history[:, 3]

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True, constrained_layout=True)
axes[0].plot(time_ms, stimulus_hz, color='black')
axes[0].set_ylabel('Estímulo (Hz)')
axes[1].plot(time_ms, E_hz, label='E')
axes[1].plot(time_ms, I_hz, label='I')
axes[1].set_ylabel('Tasa (Hz)')
axes[1].legend()
axes[2].plot(time_ms, W_e_pA, label='W_e')
axes[2].plot(time_ms, W_i_pA, label='W_i')
axes[2].set_ylabel('Adaptación (pA)')
axes[2].set_xlabel('Tiempo (ms)')
axes[2].legend()
for axis in axes:
    axis.grid(alpha=0.25)
plt.show()

print(f'E máxima: {E_hz.max():.3f} Hz | I máxima: {I_hz.max():.3f} Hz')

## 7. Guardado binario por run

El NPZ sólo contiene arrays NumPy y se puede volver a abrir con `allow_pickle=False`. Cada run guarda un experimento completo en su propio fichero.

In [ ]:
project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'src' / 'zerlaut').is_dir()
)
base_path = project_root / 'results' / 'simple_simulation.npz'
output_path = base_path.with_name(f'{base_path.stem}_run_{RUN}{base_path.suffix}')
output_path.parent.mkdir(parents=True, exist_ok=True)

result = {
    'time_ms': time_ms, 'stimulus_hz': stimulus_hz,
    'Fe_ext_hz': fe_ext_hz, 'Fi_ext_hz': fi_ext_hz,
    'E_hz': E_hz, 'I_hz': I_hz,
    'W_e_pA': W_e_pA, 'W_i_pA': W_i_pA,
}
config = {
    'simulation': {
        'run': RUN, 'duration_ms': DURATION_MS, 'dt_ms': DT_MS, 'method': 'heun',
        'initial_state': {'E_hz': 0.0, 'I_hz': 0.0, 'W_e_pA': 0.0, 'W_i_pA': 0.0},
    },
    'model': {'name': 'ZerlautAdaptationFirstOrder', 'parameters': model_parameters},
    'external_drive': {'scale': INPUT_SCALE, 'Fi_ratio': FI_RATIO},
    'stimulus': {
        'type': 'pulse', 'baseline_hz': BASELINE_HZ,
        'amplitude_hz': PULSE_AMPLITUDE_HZ, 'onset_ms': PULSE_ONSET_MS,
        'duration_ms': PULSE_DURATION_MS,
    },
    'output': {'path': str(output_path), 'dtype': 'float32'},
}
metadata = {'config': config}

arrays = {name: np.asarray(values, dtype=np.float32) for name, values in result.items()}
np.savez(output_path, **arrays, metadata_json=np.array(json.dumps(metadata)))
print(f'Run {RUN} guardada en {output_path.resolve()}')